# 🎓 AI SCHOLARSHIP FINDER AGENT
### Complete RAG Agent in Google Colab with Groq, LangChain, HuggingFace, FAISS, and Gradio

**Two Ways to Run This Notebook:**
1. **⚡ Option 1 (Fastest):** Run Cell 1 below to launch the entire agent in a single shot!
2. **🔍 Option 2 (Step-by-Step):** Or scroll down to run Steps 1 through 9 sequentially.


## ⚡ OPTION 1: ALL-IN-ONE MASTER CELL (Run Directly)
Execute this cell to install dependencies, generate the PDF, index the vector database, and launch Gradio with public sharing in one click:


In [ ]:
"""
=============================================================================
🎓 AI SCHOLARSHIP FINDER AGENT - ALL-IN-ONE MASTER GOOGLE COLAB SCRIPT
=============================================================================
Run this single cell in Google Colab to install dependencies, generate the PDF,
initialize Groq + LangChain + HuggingFace + FAISS, and launch the Gradio web UI!
"""

# 1. Install packages
!pip install -q --upgrade groq gradio langchain langchain-community langchain-core langchain-groq langchain-huggingface sentence-transformers faiss-cpu pypdf reportlab tavily-python

import os
import sys
import json
from groq import Groq
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from tavily import TavilyClient
import gradio as gr

# Try HuggingFace Embeddings
try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS

# 2. Configure API Keys
os.environ["GROQ_API_KEY"] = "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"
os.environ["TAVILY_API_KEY"] = "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"
print("✓ API Keys configured successfully.")

# 3. Create or verify PDF Knowledgebase
pdf_filename = "scholarships_database.pdf"
if not os.path.exists(pdf_filename):
    print("Generating scholarships_database.pdf...")
    doc = SimpleDocTemplate(pdf_filename, pagesize=letter)
    styles = getSampleStyleSheet()
    title_style = ParagraphStyle('DocTitle', parent=styles['Heading1'], fontSize=18, spaceAfter=14)
    heading_style = ParagraphStyle('ItemTitle', parent=styles['Heading2'], fontSize=13, textColor='#1e293b', spaceBefore=12, spaceAfter=6)
    body_style = styles['Normal']

    story = [
        Paragraph("GLOBAL SCHOLARSHIP DIRECTORY 2026-2027", title_style),
        Paragraph("Official Knowledge Base for AI SCHOLARSHIP FINDER AGENT with Eligibility, Criteria, and Funding.", body_style),
        Spacer(1, 15)
    ]

    scholarships = [
        {
            "name": "Global STEM Innovators Excellence Award",
            "provider": "International Science & Tech Foundation",
            "amount": "$25,000 / year (Renewable for 4 years)",
            "deadline": "November 15, 2026",
            "min_gpa": "3.5",
            "level": "Undergraduate",
            "fields": "Computer Science, Artificial Intelligence, Data Science, Electrical Engineering",
            "countries": "Global (All Nationalities)",
            "desc": "Designed for ambitious undergraduate students pursuing cutting-edge studies in artificial intelligence, computer science, and engineering disciplines.",
            "coverage": "Full tuition coverage, $1,500 annual tech/book stipend, and 1-on-1 industry mentorship."
        },
        {
            "name": "Ada Lovelace Women in Tech Fellowship",
            "provider": "Women Who Code & Future Leaders Alliance",
            "amount": "$20,000 lump sum + Conference Travel Grant",
            "deadline": "December 1, 2026",
            "min_gpa": "3.2",
            "level": "All (Undergraduate / Graduate)",
            "fields": "Software Engineering, Cybersecurity, Human-Computer Interaction, Robotics",
            "countries": "United States, Canada, India, United Kingdom, European Union",
            "desc": "Empowering female-identifying and non-binary students in computer science and software engineering.",
            "coverage": "Tuition aid, paid attendance at the Grace Hopper Celebration, and corporate internship fast-track."
        },
        {
            "name": "NextGen AI & Machine Learning Research Grant",
            "provider": "Open Intelligence Research Institute",
            "amount": "$35,000 research stipend + $5,000 GPU Compute",
            "deadline": "January 15, 2027",
            "min_gpa": "3.7",
            "level": "Graduate (Master's / PhD)",
            "fields": "Artificial Intelligence, Machine Learning, Computational Linguistics, Bioinformatics",
            "countries": "Global (All Nationalities)",
            "desc": "Supporting Master's and Doctoral students conducting breakthrough research in generative models.",
            "coverage": "Direct research stipend paid to student, cloud compute credits on high-performance GPU clusters."
        },
        {
            "name": "Horizon Need-Based Academic Opportunity Scholarship",
            "provider": "Civic Education Access Initiative",
            "amount": "Up to $18,000 / year (Need-based)",
            "deadline": "October 30, 2026",
            "min_gpa": "2.8",
            "level": "Undergraduate",
            "fields": "All Fields of Study (Engineering, Business, Liberal Arts, Health)",
            "countries": "United States, India, Kenya, Brazil, Philippines",
            "desc": "Ensuring that financial barriers never halt an exceptional mind. Awarded to high-potential students from lower income backgrounds.",
            "coverage": "Direct university tuition credit, campus housing support, and textbook voucher."
        }
    ]

    for item in scholarships:
        story.append(Paragraph(f"<b>Program:</b> {item['name']} ({item['provider']})", heading_style))
        content = (
            f"<b>Award:</b> {item['amount']} | <b>Deadline:</b> {item['deadline']}<br/>"
            f"<b>Min GPA:</b> {item['min_gpa']} | <b>Level:</b> {item['level']}<br/>"
            f"<b>Fields:</b> {item['fields']}<br/>"
            f"<b>Eligible Countries:</b> {item['countries']}<br/>"
            f"<b>Details:</b> {item['desc']}<br/>"
            f"<b>Coverage:</b> {item['coverage']}<br/>"
        )
        story.append(Paragraph(content, body_style))
        story.append(Spacer(1, 10))

    doc.build(story)
    print("✓ 'scholarships_database.pdf' generated successfully.")

# 4. Load & Chunk PDF
try:
    loader = PyPDFLoader(pdf_filename)
    docs = loader.load()
except Exception:
    import pypdf
    reader = pypdf.PdfReader(pdf_filename)
    docs = [Document(page_content=p.extract_text(), metadata={"page": i}) for i, p in enumerate(reader.pages)]

splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=120)
chunks = splitter.split_documents(docs)
print(f"✓ Split PDF into {len(chunks)} chunks.")

# 5. Embeddings & Vector Database
print("Initializing HuggingFace Embeddings (all-MiniLM-L6-v2)...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs={'device': 'cpu'})
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("✓ FAISS Vector Database initialized.")

# 6. Define Tools
@tool
def search_scholarship_database(query: str) -> str:
    """Searches the verified scholarship handbook PDF for eligibility, awards, deadlines, and criteria."""
    try:
        results = retriever.invoke(query)
        return "\n\n".join([f"[Chunk {i+1}]: {d.page_content.strip()}" for i, d in enumerate(results)])
    except Exception as e:
        return f"RAG search error: {str(e)}"

@tool
def calculate_education_budget(tuition_per_year: float, living_cost_per_year: float, scholarship_award_per_year: float) -> str:
    """Calculates student net out-of-pocket costs, monthly expenses, and coverage percentage."""
    try:
        t_val = float(tuition_per_year)
        l_val = float(living_cost_per_year)
        a_val = float(scholarship_award_per_year)
        total_cost = t_val + l_val
        net_cost = max(0.0, total_cost - a_val)
        coverage_pct = min(100.0, (a_val / total_cost * 100)) if total_cost > 0 else 0.0
        monthly = net_cost / 12.0
        surplus = max(0.0, a_val - total_cost)
        
        breakdown = {
            "annual_gross_expenses": "$" + f"{total_cost:,.2f}",
            "scholarship_grant": "$" + f"{a_val:,.2f}",
            "scholarship_coverage_ratio": f"{coverage_pct:.1f}%",
            "net_annual_out_of_pocket": "$" + f"{net_cost:,.2f}",
            "estimated_monthly_student_cost": "$" + f"{monthly:,.2f}",
            "stipend_surplus": ("$" + f"{surplus:,.2f}") if surplus > 0 else "None",
            "verdict": "Fully Funded!" if surplus > 0 else ("Highly Subsidized!" if coverage_pct >= 60 else "Aid gap remains.")
        }
        return json.dumps(breakdown, indent=2)
    except Exception as e:
        return f"Budget error: {str(e)}"

@tool
def search_live_web_scholarships(query: str) -> str:
    """Searches the live web for the latest 2026/2027 scholarship announcements."""
    try:
        tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
        res = tavily.search(query=query, max_results=3)
        return "\n\n".join([f"Title: {r.get('title')}\nURL: {r.get('url')}\n{r.get('content')}" for r in res.get("results", [])])
    except Exception as e:
        return f"Web search note: {str(e)}"

tools = [search_scholarship_database, calculate_education_budget, search_live_web_scholarships]
tools_by_name = {t.name: t for t in tools}

# 7. Initialize Groq LLM & Agent Loop
llm = ChatGroq(groq_api_key=os.environ["GROQ_API_KEY"], model_name="llama-3.3-70b-versatile", temperature=0.2)
llm_with_tools = llm.bind_tools(tools)

SYSTEM_PROMPT = """You are the AI SCHOLARSHIP FINDER AGENT.
Help students identify matching scholarships, verify strict eligibility criteria (GPA, major, nationality, degree level), and calculate clear financial budgets.
Always search the scholarship database first, use the budget calculator when financial numbers are mentioned, and format findings clearly."""

def run_agent(user_prompt: str, chat_history=None) -> str:
    messages = [SystemMessage(content=SYSTEM_PROMPT)]
    if chat_history:
        for item in chat_history:
            if isinstance(item, (list, tuple)) and len(item) == 2:
                if item[0]: messages.append(HumanMessage(content=str(item[0])))
                if item[1]: messages.append(AIMessage(content=str(item[1])))
            elif isinstance(item, dict):
                r = item.get("role", "")
                c = item.get("content", "")
                if r == "user" and c: messages.append(HumanMessage(content=str(c)))
                elif r == "assistant" and c: messages.append(AIMessage(content=str(c)))
                
    messages.append(HumanMessage(content=str(user_prompt)))
    
    try:
        ai_msg = llm_with_tools.invoke(messages)
        messages.append(ai_msg)
        if getattr(ai_msg, "tool_calls", None):
            for tc in ai_msg.tool_calls:
                t_name = tc.get("name") if isinstance(tc, dict) else getattr(tc, "name", "")
                t_args = tc.get("args") if isinstance(tc, dict) else getattr(tc, "args", {})
                t_id = tc.get("id", "call_1") if isinstance(tc, dict) else getattr(tc, "id", "call_1")
                if t_name in tools_by_name:
                    res = tools_by_name[t_name].invoke(t_args)
                    messages.append(ToolMessage(content=str(res), tool_call_id=t_id))
            return llm.invoke(messages).content
        return ai_msg.content
    except Exception as e:
        return f"Agent response generated: {str(e)}"

# 8. Launch Gradio UI
def gradio_chat(user_msg, hist):
    if not user_msg or not str(user_msg).strip():
        return "", hist
    if hist is None:
        hist = []
    reply = run_agent(user_msg, chat_history=hist)
    if hist and isinstance(hist[0], dict):
        hist.append({"role": "user", "content": user_msg})
        hist.append({"role": "assistant", "content": reply})
    else:
        hist.append((user_msg, reply))
    return "", hist

def gradio_budget(t, l, a):
    res = calculate_education_budget.invoke({"tuition_per_year": float(t), "living_cost_per_year": float(l), "scholarship_award_per_year": float(a)})
    return f"```json\n{res}\n```"

with gr.Blocks(theme=gr.themes.Soft(primary_hue="indigo"), title="AI SCHOLARSHIP FINDER AGENT") as demo:
    gr.Markdown("# 🎓 AI SCHOLARSHIP FINDER AGENT\n### Google Colab Live Interface (Groq + LangChain + FAISS + Tavily)")
    with gr.Tab("💬 Scholarship Chat"):
        chatbot = gr.Chatbot(height=450)
        msg = gr.Textbox(placeholder="Ask about scholarships by GPA, major, or degree level...", label="Your Question")
        with gr.Row():
            send_btn = gr.Button("Search Scholarships", variant="primary")
            clear_btn = gr.Button("Clear")
        send_btn.click(gradio_chat, inputs=[msg, chatbot], outputs=[msg, chatbot])
        msg.submit(gradio_chat, inputs=[msg, chatbot], outputs=[msg, chatbot])
        clear_btn.click(lambda: [], None, chatbot)
    with gr.Tab("💰 Budget Calculator"):
        with gr.Row():
            t_box = gr.Number(value=24000, label="Annual Tuition ($)")
            l_box = gr.Number(value=12000, label="Annual Living ($)")
            a_box = gr.Number(value=20000, label="Scholarship Aid ($)")
        b_btn = gr.Button("Compute Financial Breakdown", variant="primary")
        b_out = gr.Markdown()
        b_btn.click(gradio_budget, inputs=[t_box, l_box, a_box], outputs=b_out)

print("Starting Gradio Web Server with public share link...")
demo.launch(share=True, debug=True)




---
## 🔍 OPTION 2: STEP-BY-STEP CELLS
Run these individual cells one by one if you want to inspect each step:


### Step 1: Install Dependencies in Google Colab
Installs Groq, Gradio, LangChain, HuggingFace embeddings, FAISS/Chroma, PyPDF, ReportLab, and Tavily without dependency conflicts.


In [ ]:
!pip install -q --upgrade groq gradio langchain langchain-community langchain-core langchain-groq langchain-huggingface sentence-transformers faiss-cpu pypdf reportlab tavily-python

print("All dependencies installed cleanly! Ready to launch AI Scholarship Finder Agent.")


### Step 2: Auto-Generate or Verify Scholarship PDF Database
Creates 'scholarships_database.pdf' using Python ReportLab if not already uploaded, preventing missing-file errors.


In [ ]:
import os
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle

pdf_filename = "scholarships_database.pdf"

# If the user has not uploaded a custom PDF, generate a verified rich scholarship database
if not os.path.exists(pdf_filename):
    print(f"Generating verified '{pdf_filename}' for RAG Knowledgebase...")
    doc = SimpleDocTemplate(pdf_filename, pagesize=letter)
    styles = getSampleStyleSheet()
    
    title_style = ParagraphStyle(
        'DocTitle',
        parent=styles['Heading1'],
        fontSize=18,
        spaceAfter=14
    )
    heading_style = ParagraphStyle(
        'ItemTitle',
        parent=styles['Heading2'],
        fontSize=13,
        textColor='#1e293b',
        spaceBefore=12,
        spaceAfter=6
    )
    body_style = styles['Normal']

    story = [
        Paragraph("GLOBAL SCHOLARSHIP DIRECTORY 2026-2027", title_style),
        Paragraph("Official Knowledge Base for AI SCHOLARSHIP FINDER AGENT with Eligibility, Criteria, and Funding.", body_style),
        Spacer(1, 15)
    ]

    scholarships = [
        {
            "name": "Global STEM Innovators Excellence Award",
            "provider": "International Science & Tech Foundation",
            "amount": "$25,000 / year (Renewable for 4 years)",
            "deadline": "November 15, 2026",
            "min_gpa": "3.5",
            "level": "Undergraduate",
            "fields": "Computer Science, Artificial Intelligence, Data Science, Electrical Engineering",
            "countries": "Global (All Nationalities)",
            "desc": "Designed for ambitious undergraduate students pursuing cutting-edge studies in artificial intelligence, computer science, and engineering disciplines with demonstrated leadership and technical potential.",
            "coverage": "Full tuition coverage, $1,500 annual tech/book stipend, and 1-on-1 industry mentorship."
        },
        {
            "name": "Ada Lovelace Women in Tech Fellowship",
            "provider": "Women Who Code & Future Leaders Alliance",
            "amount": "$20,000 lump sum + Conference Travel Grant",
            "deadline": "December 1, 2026",
            "min_gpa": "3.2",
            "level": "All (Undergraduate / Graduate)",
            "fields": "Software Engineering, Cybersecurity, Human-Computer Interaction, Robotics",
            "countries": "United States, Canada, India, United Kingdom, European Union",
            "desc": "Empowering female-identifying and non-binary students in computer science and software engineering to bridge the gender disparity in technology leadership roles.",
            "coverage": "Tuition aid, paid attendance at the Grace Hopper Celebration, and corporate internship fast-track."
        },
        {
            "name": "NextGen AI & Machine Learning Research Grant",
            "provider": "Open Intelligence Research Institute",
            "amount": "$35,000 research stipend + $5,000 GPU Compute",
            "deadline": "January 15, 2027",
            "min_gpa": "3.7",
            "level": "Graduate (Master's / PhD)",
            "fields": "Artificial Intelligence, Machine Learning, Computational Linguistics, Bioinformatics",
            "countries": "Global (All Nationalities)",
            "desc": "Supporting Master's and Doctoral students conducting breakthrough research in generative models, NLP, reinforcement learning, and open-source scientific computing.",
            "coverage": "Direct research stipend paid to student, cloud compute credits on high-performance GPU clusters."
        },
        {
            "name": "Horizon Need-Based Academic Opportunity Scholarship",
            "provider": "Civic Education Access Initiative",
            "amount": "Up to $18,000 / year (Need-based)",
            "deadline": "October 30, 2026",
            "min_gpa": "2.8",
            "level": "Undergraduate",
            "fields": "All Fields of Study (Engineering, Business, Liberal Arts, Health)",
            "countries": "United States, India, Kenya, Brazil, Philippines",
            "desc": "Ensuring that financial barriers never halt an exceptional mind. Awarded to high-potential students from low-to-middle income backgrounds across all academic majors.",
            "coverage": "Direct university tuition credit, campus housing support, and textbook voucher."
        },
        {
            "name": "Turing Postgraduate Data Science Excellence Scholarship",
            "provider": "European Analytics Consortium",
            "amount": "€22,000 / year + Full Tuition Waiver",
            "deadline": "February 28, 2027",
            "min_gpa": "3.6",
            "level": "Graduate (Master of Science)",
            "fields": "Data Science, Statistics, Quantitative Finance, Applied Mathematics",
            "countries": "Global (All Nationalities)",
            "desc": "Geared towards graduate candidates pursuing Master of Science in Data Analytics or Quantitative Modeling with strong coding skills in Python, R, or Julia.",
            "coverage": "Full European university tuition waiver plus €1,800 monthly living allowance."
        }
    ]

    for item in scholarships:
        story.append(Paragraph(f"<b>Program:</b> {item['name']} ({item['provider']})", heading_style))
        content = (
            f"<b>Award:</b> {item['amount']} | <b>Deadline:</b> {item['deadline']}<br/>"
            f"<b>Min GPA:</b> {item['min_gpa']} | <b>Level:</b> {item['level']}<br/>"
            f"<b>Fields:</b> {item['fields']}<br/>"
            f"<b>Eligible Countries:</b> {item['countries']}<br/>"
            f"<b>Details:</b> {item['desc']}<br/>"
            f"<b>Coverage:</b> {item['coverage']}<br/>"
        )
        story.append(Paragraph(content, body_style))
        story.append(Spacer(1, 10))

    doc.build(story)
    print(f"'{pdf_filename}' created successfully ({len(scholarships)} programs indexed)!")
else:
    print(f"Using existing '{pdf_filename}' in workspace.")


### Step 3: Configure Groq API Key & Test Connection (with Auto-Model Fallback)
Sets up Groq and Tavily credentials, tests connectivity, and handles model fallback if openai/gpt-oss-120h is queried.


In [ ]:
import os
from groq import Groq

# Configure your keys
os.environ["GROQ_API_KEY"] = "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"
os.environ["TAVILY_API_KEY"] = "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"

print("API Keys configured successfully.")

client = Groq(api_key=os.environ["GROQ_API_KEY"])

# Resolve model cleanly (handles openai/gpt-oss-120h or standard Groq models)
preferred_model = "llama-3.3-70b-versatile"
test_models = ["openai/gpt-oss-120h", "llama-3.3-70b-versatile", "llama-3.1-8b-instant"]

active_model = None
for m in test_models:
    try:
        print(f"Testing Groq model: '{m}'...")
        completion = client.chat.completions.create(
            model=m,
            messages=[
                {"role": "system", "content": "You are the AI SCHOLARSHIP FINDER AGENT. Respond in one cheerful sentence."},
                {"role": "user", "content": "Hello! Confirm you are ready to find scholarships."}
            ],
            max_tokens=60,
            temperature=0.3
        )
        active_model = m
        print(f"Success with model: '{active_model}'!")
        print(f"Agent response: {completion.choices[0].message.content}\n")
        break
    except Exception as e:
        print(f"Notice: Model '{m}' returned error: {e}. Trying next available Groq model...")

if not active_model:
    active_model = "llama-3.3-70b-versatile"

print(f"Active Groq Model selected: '{active_model}'")


### Step 4: Load PDF & Split into Chunks with LangChain
Loads the PDF using PyPDFLoader and splits it into semantic chunks with RecursiveCharacterTextSplitter.


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

pdf_path = "scholarships_database.pdf"

try:
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    print(f"Successfully loaded {len(documents)} page(s) from '{pdf_path}'.")
except Exception as e:
    print(f"PyPDFLoader notice ({e}), loading with fallback document reader...")
    from langchain_core.documents import Document
    import pypdf
    reader = pypdf.PdfReader(pdf_path)
    documents = [Document(page_content=p.extract_text(), metadata={"page": i}) for i, p in enumerate(reader.pages)]
    print(f"Extracted {len(documents)} page(s).")

# Split into chunks using RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=120,
    separators=["\n\n", "\n", " ", ""]
)

chunks = text_splitter.split_documents(documents)
print(f"Generated {len(chunks)} text chunks for embedding.")
print(f"Preview of Chunk #1: {chunks[0].page_content[:150]}...")


### Step 5: Create HuggingFace Embeddings & Vector Database
Embeds chunks using all-MiniLM-L6-v2 and stores them in FAISS/Chroma with automatic Colab SQLite compatibility.


In [ ]:
import sys

# 1. Initialize HuggingFace Embeddings
try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings

print("Initializing HuggingFace Embeddings ('all-MiniLM-L6-v2')...")
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

# 2. Build Vector Store (FAISS / Chroma fallback for 100% zero-crash compatibility in Colab)
vectorstore = None

try:
    from langchain_community.vectorstores import FAISS
    vectorstore = FAISS.from_documents(chunks, embeddings)
    print("Vector database: FAISS initialized successfully (fast in-memory indexing)!")
except Exception as faiss_err:
    print(f"FAISS notice ({faiss_err}), initializing Chroma...")
    from langchain_community.vectorstores import Chroma
    vectorstore = Chroma.from_documents(chunks, embeddings)
    print("Vector database: Chroma initialized successfully!")

# 3. Create retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("Retriever configured and ready for queries!")


### Step 6: Test RAG Semantic Retrieval
Executes a test query against the vector store to verify semantic document retrieval.


In [ ]:
test_query = "scholarships for female students in computer science and software"
retrieved_docs = retriever.invoke(test_query)

print(f"Query: '{test_query}'\n")
print(f"Retrieved {len(retrieved_docs)} relevant context document(s):\n")

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"--- Document #{i} ---")
    print(doc.page_content.strip()[:250] + "...")
    print("-" * 40)


### Step 7: Build RAG Tool, Budget Calculator Tool, and Web Search Tool
Creates specialized agent tools: PDF knowledgebase retriever, net tuition budget calculator, and live Tavily web search.


In [ ]:
import json
from langchain_core.tools import tool
from tavily import TavilyClient

# --- Tool 1: RAG Scholarship Database Search ---
@tool
def search_scholarship_database(query: str) -> str:
    """Searches the verified scholarship handbook PDF for eligibility, awards, deadlines, and criteria."""
    try:
        docs = retriever.invoke(query)
        if not docs:
            return "No specific match found in the local scholarship PDF database."
        results = []
        for i, doc in enumerate(docs, 1):
            results.append(f"[Document Chunk {i}]:\n{doc.page_content.strip()}")
        return "\n\n".join(results)
    except Exception as e:
        return f"Database query note: {str(e)}"

# --- Tool 2: Financial Aid & Budget Calculator Tool ---
@tool
def calculate_education_budget(tuition_per_year: float, living_cost_per_year: float, scholarship_award_per_year: float) -> str:
    """Calculates student net out-of-pocket costs, monthly expenses, and coverage percentage.
    Inputs:
      tuition_per_year: Annual tuition and academic fees in USD.
      living_cost_per_year: Estimated housing, food, transportation per year.
      scholarship_award_per_year: Total scholarship or grant amount received per year.
    """
    try:
        t_val = float(tuition_per_year)
        l_val = float(living_cost_per_year)
        a_val = float(scholarship_award_per_year)
        
        total_cost = t_val + l_val
        net_cost = max(0.0, total_cost - a_val)
        coverage_pct = min(100.0, (a_val / total_cost * 100)) if total_cost > 0 else 0.0
        monthly_out_of_pocket = net_cost / 12.0
        surplus = max(0.0, a_val - total_cost)

        breakdown = {
            "annual_gross_expenses": "$" + f"{total_cost:,.2f}",
            "scholarship_grant": "$" + f"{a_val:,.2f}",
            "scholarship_coverage_ratio": f"{coverage_pct:.1f}%",
            "net_annual_out_of_pocket": "$" + f"{net_cost:,.2f}",
            "estimated_monthly_student_cost": "$" + f"{monthly_out_of_pocket:,.2f}",
            "stipend_surplus": ("$" + f"{surplus:,.2f}") if surplus > 0 else "None (Costs remain)",
            "financial_verdict": (
                "Fully Funded! You have excess funds." if surplus > 0 else
                "Highly Subsidized! Manageable out-of-pocket balance." if coverage_pct >= 60 else
                "Significant gap remains. Additional aid or part-time work recommended."
            )
        }
        return json.dumps(breakdown, indent=2)
    except Exception as e:
        return f"Budget calculation error: {str(e)}"

# --- Tool 3: Live Web Search via Tavily ---
@tool
def search_live_web_scholarships(query: str) -> str:
    """Searches the live web for the latest 2026/2027 scholarship announcements, international government grants, and university deadlines."""
    try:
        tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
        res = tavily.search(query=query, max_results=3)
        formatted = []
        for r in res.get("results", []):
            formatted.append(f"Title: {r.get('title')}\nURL: {r.get('url')}\nSnippet: {r.get('content')}")
        return "\n\n".join(formatted) if formatted else "No live web results returned."
    except Exception as e:
        return f"Live search note: {str(e)}"

tools = [search_scholarship_database, calculate_education_budget, search_live_web_scholarships]
print(f"Created {len(tools)} specialized agent tools: {[t.name for t in tools]}")


### Step 8: Connect LangChain to Groq & Initialize Resilient Agent
Initializes ChatGroq, binds tools, handles multi-format chat history, and executes tool routing safely.


In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

# Initialize Groq LLM with LangChain
llm = ChatGroq(
    groq_api_key=os.environ["GROQ_API_KEY"],
    model_name="llama-3.3-70b-versatile",
    temperature=0.2,
    max_tokens=1024
)

# Bind tools to Groq model
llm_with_tools = llm.bind_tools(tools)
tools_by_name = {tool.name: tool for tool in tools}

SYSTEM_PROMPT = """You are the AI SCHOLARSHIP FINDER AGENT.
Your mission is to help students identify matching scholarships, verify eligibility criteria (GPA, major, nationality, degree level), and calculate clear financial budgets.

Guidelines:
1. Search the local database using 'search_scholarship_database' for verified scholarships.
2. If the user asks for broader or external opportunities, use 'search_live_web_scholarships'.
3. When the user mentions tuition, living expenses, or awards, use 'calculate_education_budget'.
4. Format responses clearly with Program Name, Provider, Award, Minimum GPA, Eligibility, and Next Steps.
Be encouraging, precise, and helpful!"""

def run_agent(user_prompt: str, chat_history=None) -> str:
    """Executes the AI SCHOLARSHIP FINDER AGENT loop with tool calling and history support."""
    messages = [SystemMessage(content=SYSTEM_PROMPT)]
    
    # Safely handle both tuple list [(u, a)] and dict list [{'role':'user',...}] from Gradio
    if chat_history:
        for item in chat_history:
            if isinstance(item, (list, tuple)) and len(item) == 2:
                u, a = item
                if u: messages.append(HumanMessage(content=str(u)))
                if a: messages.append(AIMessage(content=str(a)))
            elif isinstance(item, dict):
                r = item.get("role", "")
                c = item.get("content", "")
                if r == "user" and c: messages.append(HumanMessage(content=str(c)))
                elif r == "assistant" and c: messages.append(AIMessage(content=str(c)))
            
    messages.append(HumanMessage(content=str(user_prompt)))
    
    try:
        # First model call
        ai_msg = llm_with_tools.invoke(messages)
        messages.append(ai_msg)
        
        # Handle tool calls if any
        if getattr(ai_msg, "tool_calls", None):
            for tool_call in ai_msg.tool_calls:
                t_name = tool_call.get("name") if isinstance(tool_call, dict) else getattr(tool_call, "name", "")
                t_args = tool_call.get("args") if isinstance(tool_call, dict) else getattr(tool_call, "args", {})
                t_id = tool_call.get("id", "call_1") if isinstance(tool_call, dict) else getattr(tool_call, "id", "call_1")
                
                if t_name in tools_by_name:
                    selected_tool = tools_by_name[t_name]
                    try:
                        tool_output = selected_tool.invoke(t_args)
                    except Exception as err:
                        tool_output = f"Tool notice: {str(err)}"
                    messages.append(ToolMessage(content=str(tool_output), tool_call_id=t_id))
            
            # Second call to synthesize tool findings
            final_response = llm.invoke(messages)
            return final_response.content
        
        return ai_msg.content
    except Exception as e:
        return f"I encountered a temporary processing note: {str(e)}. Please check your query or API quota."

# Test the agent with a sample query
print("Testing AI SCHOLARSHIP FINDER AGENT...")
test_reply = run_agent("I am a Computer Science student with a 3.6 GPA. What scholarships can I apply for?")
print("\n[Agent Output Preview]:")
print(test_reply[:400] + "...")


### Step 9: Launch Gradio Web Interface in Google Colab
Creates an interactive Gradio UI with chat, quick matcher, and budget calculator, generating a live shareable public URL.


In [ ]:
import gradio as gr

def gradio_chat(user_message, history):
    if not user_message or not str(user_message).strip():
        return "", history
    if history is None:
        history = []
        
    bot_reply = run_agent(user_message, chat_history=history)
    
    # Handle Gradio versions with dict or tuple histories
    if history and isinstance(history[0], dict):
        history.append({"role": "user", "content": user_message})
        history.append({"role": "assistant", "content": bot_reply})
    else:
        history.append((user_message, bot_reply))
    return "", history

def gradio_budget(tuition, living, award):
    result_str = calculate_education_budget.invoke({
        "tuition_per_year": float(tuition),
        "living_cost_per_year": float(living),
        "scholarship_award_per_year": float(award)
    })
    try:
        data = json.loads(result_str)
        output = f"""### 📊 Student Budget & Out-of-Pocket Breakdown:
- **Annual Total Expenses:** {data['annual_gross_expenses']}
- **Scholarship Aid:** {data['scholarship_grant']}
- **Coverage Ratio:** {data['scholarship_coverage_ratio']}
- **Net Out-of-Pocket Annual Cost:** {data['net_annual_out_of_pocket']}
- **Estimated Monthly Burden:** {data['estimated_monthly_student_cost']}
- **Stipend Surplus:** {data['stipend_surplus']}

**Verdict:** {data['financial_verdict']}
"""
        return output
    except Exception:
        return result_str

def gradio_quick_match(major, gpa, level):
    query = f"Find scholarship for a {level} student majoring in {major} with a GPA of {gpa}"
    return run_agent(query)

# Build Gradio UI
with gr.Blocks(theme=gr.themes.Soft(primary_hue="indigo"), title="AI SCHOLARSHIP FINDER AGENT") as demo:
    gr.Markdown("""
    # 🎓 AI SCHOLARSHIP FINDER AGENT
    ### Powered by **LangChain + Groq (LLaMA-3.3-70B) + HuggingFace Embeddings + FAISS / Chroma + Tavily**
    Ask questions about scholarships, match your GPA & major, or calculate your out-of-pocket tuition costs!
    """)
    
    with gr.Tab("💬 Scholarship Finder Chat"):
        chatbot = gr.Chatbot(height=450, label="Agent Conversation")
        msg = gr.Textbox(placeholder="Ask anything (e.g. 'Scholarships for AI/ML graduate students', 'Undergraduate STEM grants')...", label="Your Query")
        with gr.Row():
            send_btn = gr.Button("🔍 Search & Analyze", variant="primary")
            clear_btn = gr.Button("🗑️ Clear Chat")
            
        send_btn.click(gradio_chat, inputs=[msg, chatbot], outputs=[msg, chatbot])
        msg.submit(gradio_chat, inputs=[msg, chatbot], outputs=[msg, chatbot])
        clear_btn.click(lambda: [], None, chatbot, queue=False)
        
    with gr.Tab("⚡ Quick Profile Matcher"):
        with gr.Row():
            major_input = gr.Textbox(value="Computer Science", label="Major / Academic Field")
            gpa_input = gr.Number(value=3.6, label="Current GPA (out of 4.0)")
            level_input = gr.Dropdown(choices=["Undergraduate", "Graduate (Master's)", "PhD", "All"], value="Undergraduate", label="Degree Level")
        match_btn = gr.Button("🎯 Match Eligible Scholarships", variant="primary")
        match_output = gr.Markdown(label="Match Recommendations")
        match_btn.click(gradio_quick_match, inputs=[major_input, gpa_input, level_input], outputs=match_output)

    with gr.Tab("💰 Education Budget Calculator"):
        with gr.Row():
            tuition = gr.Number(value=22000, label="Annual Tuition & Academic Fees ($)")
            living = gr.Number(value=10000, label="Annual Housing & Living Costs ($)")
            award = gr.Number(value=20000, label="Scholarship Grant ($)")
        calc_btn = gr.Button("Calculate Net Cost & Coverage", variant="primary")
        budget_output = gr.Markdown()
        calc_btn.click(gradio_budget, inputs=[tuition, living, award], outputs=budget_output)

    gr.Markdown("---  \n*AI SCHOLARSHIP FINDER AGENT - Running directly in Google Colab with Public URL Sharing*")

# Launch with share=True to get both local (127.0.0.1:7860) and public gradio.live URL in Google Colab!
demo.launch(share=True, debug=True)
